# Feature Engineering & Selection

####  List of exercises
>  - **Exercise 1**: Feature Selection with Recursive Feature Elimination (RFE).

## Feature Engineering: Titanic Dataset
Welcome! In this notebook, we’ll walk through practical examples of feature engineering using the Titanic dataset.

**Goals:**
- Load and explore the Titanic dataset
- Perform common feature engineering steps
- Train a simple model and evaluate its performance
- Visualize feature importance
- Explore stretch topics like automated feature engineering and PCA

In [0]:
# Step 1: Load Dataset
import pandas as pd
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

# import seaborn as sns
# Load the Titanic dataset
# df = sns.load_dataset('titanic')

df.head()

In [0]:
# 🧹 Step 2: Feature Engineering
# Create 'Title' from Name
df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\s*\.')

# Create 'FamilySize'
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# Fill missing Age with median
df['Age'] = df['Age'].fillna(df['Age'].median())

# Fill missing Embarked with mode
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Bin Fare
df['FareBin'] = pd.qcut(df['Fare'], 4, labels=False)

# Encode categorical variables
df_encoded = pd.get_dummies(df[['Sex', 'Embarked', 'Title']], drop_first=True)

# Combine all features
features = pd.concat([df[['Pclass', 'Age', 'Fare', 'FamilySize', 'FareBin']], df_encoded], axis=1)
target = df['Survived']

In [0]:
# Step 3: Train/Test Split and Modeling
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3}")

In [0]:
# Step 4: Feature Importance
import matplotlib.pyplot as plt
feature_names = features.columns
importances = pd.Series(clf.feature_importances_, index=X_train.columns)
_ = importances.sort_values(ascending=True).plot.barh(x='Category', y='Value', title='Feature Importance')

## RFE Visualization with Yellowbrick
- [Yellowbrick](https://www.scikit-yb.org/en/latest/index.html) extends the Scikit-Learn API to make model selection and hyperparameter tuning easier. Under the hood, it’s using `Matplotlib`.

- Recursive feature elimination (RFE) is a feature selection method that fits a model and removes the weakest feature (or features) until the specified number of features is reached. 
- Features are ranked by the model’s `coef_` or `feature_importances_` attributes, and by recursively eliminating a small number of features per loop, RFE attempts to eliminate *dependencies* and *collinearity* that may exist in the data.

- RFE requires a specified number of features to keep
- However, we often don't know in advance how many (and which) features to keep.
- Cross-validation is used with RFE to score different feature subsets and select the best scoring collection of features.
- The RFECV visualizer plots the number of features in the model along with their cross-validated test score and variability and visualizes the selected number of features.

To install, use either of the two approaches below:

- Using pip:  `pip install -U yellowbrick`
  
- If you’re using Anaconda:  `conda install -c districtdatalabs yellowbrick`
 

<br>
<div class="alert alert-info">
<b style="font-size: 25px;">Exercise 1: Feature Selection with Recursive Feature Elimination (RFE) </b>
</div>

### Objective
Use Recursive Feature Elimination (RFE) to reduce the number of input features in a classification model, compare performance, and reflect on the trade-offs between model complexity and accuracy.

You will:

- Select the most relevant features using RFE

- Compare model performance across different feature counts

- Interpret and defend your selection.

### Dataset
Use the Breast Cancer dataset `from sklearn.datasets.load_breast_cancer()`.

### Part 1: Baseline model

- Load the breast cancer dataset and split it into training and test sets.
- Build a classification model using `LogisticRegression`.
- Evaluate your baseline model with all features.
- Questions:
   - What is your baseline test accuracy?
   - Is there evidence of overfitting (compare train vs. test performance)?
### Part 2: Apply Recursive Feature Elimination

- Use `sklearn.feature_selection.RFE` to select exactly 5 features.
- Retrain your model using only these selected features.
- Compare performance to the baseline.
- Questions:
  - Which features were selected?
  - How does test performance compare to the full-feature model?
  - Are the selected features interpretable or surprising?

### Part 3: Systematic Comparison
- Use a loop (or list comprehension) to evaluate RFE with different numbers of features (from 1 up to all).
- Record accuracy at each level (you may use test set or cross-validation).
- Plot number of features vs. model accuracy.
- Questions:
  - At what point does adding more features stop helping?
  - Would you choose the number of features with the best accuracy, or fewer features with slightly lower accuracy?

### Part 4: Model Comparison 
- Try a different model
   - Use RandomForestClassifier instead of logistic regression.

   - Repeat the RFE process and analyze how the selected features differ.

- Compare RFE vs. Feature Importance Plot

  - Plot the feature importances from the trained RandomForestClassifier.

  - Select the top 5 features based on importance scores.

  - Retrain the model only on these 5 features.

  - Compare this model’s performance to the model trained with 5 features selected via RFE.

- Questions to Reflect On:

  - Do RFE and the feature importance method select the same features?

  - Which model performs better? Why might that be?

  - When might you prefer one method over the other?

> **Hint**: Use `.feature_importances_` from the trained `RandomForestClassifier` and `pandas.Series(...).nlargest(5)` to select top features.

### Part 5: RFECV 
- Automate the process and use Yellowbrick's `RFECV` (Recursive Feature Elimination with Cross-Validation) together with `StratifiedKFold` cross-validation (from scikit-learn).
- Apply using both models (i.e., logistic regression and random forests) considered above.
- 
> **Learn More**: Explore how RFECV works and how to apply it in your own models by visiting the official [scikit-yb RFECV documentation](https://www.scikit-yb.org/en/latest/api/model_selection/rfecv.html).


### Part 5: Apply to another dataset
- Try the RFE workflow on a different dataset like Diabetes or Titanic.
- How do your choices change based on the dataset?

### Submission

Your final analysis as part of this notebook. You analysis should include plots of #features vs. accuracy where applicable. In addition, add the following  
- A short paragraph (4–5 sentences) answering:
   - What model and number of features did you choose?
   - Why did you choose that model, and that number of features?
   - What trade-offs did you consider in making your decision?

<br>
<div class="alert alert-info">
<b style="font-size: 25px;">Exercise 1: End</b>
</div>